In [ ]:
import requests
requests.packages.urllib3.disable_warnings()
from bs4 import BeautifulSoup
import re
import os
import time

In [ ]:

# This function takes a BeautifulSoup object as input and extracts the values of the traits for a given species from the PALDat website. It returns a dictionary where the keys are the names of the traits and the values are the corresponding values for those traits.
def scrape_paldat_values(soup):
    
    # Extract keys and values from the page and add to a dictionary. Keys are the names of the traits, and values are the corresponding values for those traits.
    keys = [item.get_text().replace(':', '').replace('\xa0',' ') for item in soup.find_all('span', class_ = 'diag-label')]
    values = [item.get_text() for item in soup.find_all('span', class_ = 'diag-value')]
    value_dict = {key:value for key, value in zip(keys, values)}

    # Extract species and genus names. Species name contains genus name as well.
    value_dict['Species'] = soup.find(class_ = 'species').get_text()
    value_dict['Genus'] = soup.find(class_ = 'genus').get_text()
    
    # Parse taxonomy levels and add to value_dict. This will allow for analysis by higher taxonomic levels, should species level prove too complex.
    taxonomy = soup.find('p').get_text().replace('Taxonomy: ','').replace(',','').splitlines()
    tax_dict = {}
    taxonomy_levels = ['Family', 'Order', 'Class', 'Phylum']
    for item, level in zip(taxonomy, taxonomy_levels):
        tax_dict[level] = taxonomy[len(taxonomy) - taxonomy.index(item) - 2]
    value_dict.update(tax_dict)

    return value_dict

c:\Users\pinto\.virtualenvs\pollen-main-gUuID1Jd\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.paldat.org'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


{'pollen unit': 'monad', 'dispersal unit and peculiarities': 'monad', 'size (pollen unit)': 'medium-sized (26-50 µm)', 'size of hydrated pollen (LM)': '41-50 µm,  51-100 µm', 'shortest polar axis in equatorial view (LM)': '36-40 µm', 'longest polar axis in equatorial view (LM)': '41-50 µm', 'shortest diameter in equatorial or polar view (LM)': '41-50 µm', 'longest diameter in equatorial or polar view (LM)': '51-100 µm', 'pollen class': 'colpate', 'polarity': 'isopolar', 'P/E-ratio': 'prolate', 'shape': '-', 'outline in polar view': 'triangular', 'dominant orientation (LM)': 'oblique', 'P/E-ratio (dry pollen)': '-', 'shape (dry pollen)': '-', 'outline in polar view (dry pollen)': '-', 'infoldings (dry pollen)': '-', 'aperture number': '3', 'aperture type': 'colpus', 'aperture condition': 'colpate,  tricolpate', 'aperture peculiarities': 'brevicolpus', 'ornamentation LM': 'echinate', 'nexine': '-', 'sexine': '-', 'ornamentation SEM': '-', 'suprasculpture SEM': '-', 'tectum': '-', 'infrat

c:\Users\pinto\.virtualenvs\pollen-main-gUuID1Jd\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.paldat.org'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\pinto\.virtualenvs\pollen-main-gUuID1Jd\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.paldat.org'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


{'pollen unit': 'monad', 'dispersal unit and peculiarities': 'monad', 'size (pollen unit)': 'large (51-100 µm)', 'size of hydrated pollen (LM)': '51-100 µm', 'shortest polar axis in equatorial view (LM)': '31-35 µm', 'longest polar axis in equatorial view (LM)': '-', 'shortest diameter in equatorial or polar view (LM)': '51-100 µm', 'longest diameter in equatorial or polar view (LM)': '-', 'pollen class': 'sulcate', 'polarity': 'isopolar', 'P/E-ratio': 'oblate', 'shape': '-', 'outline in polar view': 'elliptic', 'dominant orientation (LM)': 'polar', 'P/E-ratio (dry pollen)': '-', 'shape (dry pollen)': '-', 'outline in polar view (dry pollen)': '-', 'infoldings (dry pollen)': '-', 'aperture number': '2', 'aperture type': 'sulcus', 'aperture condition': 'sulcate,  disulcate', 'aperture peculiarities': '-', 'ornamentation LM': 'echinate', 'nexine': '-', 'sexine': '-', 'ornamentation SEM': '-', 'suprasculpture SEM': '-', 'tectum': '-', 'infratectum': '-', 'foot layer': '-', 'endexine': '-'

c:\Users\pinto\.virtualenvs\pollen-main-gUuID1Jd\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.paldat.org'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\pinto\.virtualenvs\pollen-main-gUuID1Jd\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.paldat.org'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


In [ ]:
# Retrieve Genus page links
links = []
for letter in  [chr(i) for i in range(ord('A'), ord('Z') + 1)]:
    r = requests.get(f'https://www.paldat.org/search/{letter}', verify = False)
    soup = BeautifulSoup(r.text, features = 'lxml')
    links.extend(['https://www.paldat.org' + link.get('href') for link in soup.find_all('a', class_ = 'genus')])

# Retrieve Species page links
species_links = []
for link in links:
    r = requests.get(link, verify = False)
    soup = BeautifulSoup(r.text, features = 'lxml')
    species_links.extend(['https://www.paldat.org' + link.get('href') for link in soup.find_all('a', class_ = 'species')])

c:\Users\pinto\.virtualenvs\pollen-main-gUuID1Jd\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.paldat.org'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\pinto\.virtualenvs\pollen-main-gUuID1Jd\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.paldat.org'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\pinto\.virtualenvs\pollen-main-gUuID1Jd\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.paldat.org'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warn

['https://www.paldat.org/search/genus/Abelia;jsessionid=C6CC1F5C2EC255F29488B6C9E626D69C', 'https://www.paldat.org/search/genus/Abeliophyllum;jsessionid=C6CC1F5C2EC255F29488B6C9E626D69C', 'https://www.paldat.org/search/genus/Abies;jsessionid=C6CC1F5C2EC255F29488B6C9E626D69C', 'https://www.paldat.org/search/genus/Abutilon;jsessionid=C6CC1F5C2EC255F29488B6C9E626D69C', 'https://www.paldat.org/search/genus/Acacia;jsessionid=C6CC1F5C2EC255F29488B6C9E626D69C', 'https://www.paldat.org/search/genus/Acaena;jsessionid=C6CC1F5C2EC255F29488B6C9E626D69C', 'https://www.paldat.org/search/genus/Acalypha;jsessionid=C6CC1F5C2EC255F29488B6C9E626D69C', 'https://www.paldat.org/search/genus/Acantholimon;jsessionid=C6CC1F5C2EC255F29488B6C9E626D69C', 'https://www.paldat.org/search/genus/Acanthopanax;jsessionid=C6CC1F5C2EC255F29488B6C9E626D69C', 'https://www.paldat.org/search/genus/Acanthostachys;jsessionid=C6CC1F5C2EC255F29488B6C9E626D69C', 'https://www.paldat.org/search/genus/Acanthus;jsessionid=C6CC1F5C2EC2

c:\Users\pinto\.virtualenvs\pollen-main-gUuID1Jd\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.paldat.org'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


In [5]:
import pandas as pd

pollen_df = pd.read_csv('paldat_out.csv')

In [40]:
for column in pollen_df.columns:
    print(pollen_df[pollen_df['imgs'] != '[]'].replace('-', pd.NA)[column].value_counts())

Class
Asterales          320
Lamiales           272
Fabales            175
Rosales            133
Brassicales        119
Caryophyllales     117
Ranunculales       112
Asparagales        108
Ericales            79
Apiales             78
Malpighiales        75
Poales              67
Dipsacales          60
Gentianales         56
Malvales            55
Saxifragales        46
unplaced            40
Solanales           36
Myrtales            32
Sapindales          31
Liliales            28
Pinales             25
Fagales             25
Geraniales          16
Alismatales         11
Cornales            11
Cucurbitales        10
Santanales          10
Zingiberales         7
Celastrales          7
Zygophyllales        6
Oxalidales           5
Arecales             5
Proteales            5
Magnoliales          4
Buxales              3
Laurales             3
Vitales              3
Commelinales         2
Piperales            2
Dioscoreales         2
Gnetales             2
Crossosomatales      2
Ginkg

In [1]:
#os.mkdir('images')
values_dict = {}
id_counter = 0
for link in species_links:
    r = requests.get(link, verify = False)
    soup = BeautifulSoup(r.text, features = 'lxml')
    values = scrape_paldat_values(soup)
    values_dict[values['Species']] = values
    
    # Scrape images for each species and save to local directory. Store filenames in values_dict.
    values_dict[values['Species']]['imgs'] = []
    for link in soup.find_all('a', id ='pic_0'):
        img_data = requests.get('https://www.paldat.org' + link.get('href'), verify=False).content
        filename = f"images/{link.get('data-title')}_{id_counter}.jpg"
        with open(filename, 'wb') as f:
            f.write(img_data)
        
        values_dict[values['Species']]['imgs'].append(f'{link.get("data-title")}_{id_counter}')
        id_counter += 1
    time.sleep(3)

NameError: name 'species_links' is not defined